# Unit 1: Training a Deep Reinforcement Learning Agent 🤖

![Cover](https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/thumbnail.jpg)

## Project Overview
In this notebook, I train my **first Deep Reinforcement Learning agent**: a Lunar Lander that will learn to **land correctly on the Moon 🌕**.

Using [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/), I will train the agent, evaluate its performance, and share the results on the Hugging Face Hub.

### 🛠️ Environment & Tools

- **Environment 🎮:**
    
    [LunarLander-v3](https://gymnasium.farama.org/environments/box2d/lunar_lander/) (Gymnasium)

- **Library 📚:**

    [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/) (PPO Algorithm)

### 🏆 Notebook Objectives

In this project, I focus on the following steps:
1.  Setting up the **Gymnasium** environment.
2.  Training an agent using **Stable-Baselines3**.
3.  **Evaluating and Publishing** the trained agent to the Hub with a replay video and evaluation score 🔥.

## A small recap of Deep Reinforcement Learning 📚

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/RL_process_game.jpg" alt="The RL process" width="100%">

## 📚 Theoretical Recap

Before training the agent, here is a summary of the key Reinforcement Learning concepts covered in this unit:

* **Reinforcement Learning (RL):** A computational approach where an agent learns by **interacting with an environment** through trial and error, receiving feedback in the form of rewards (positive or negative).
* **The Goal:** To maximize the **expected cumulative reward** (also called return). This is based on the *Reward Hypothesis*: all goals can be described as the maximization of expected cumulative reward.
* **The RL Loop:** The process is a continuous cycle: **State $\rightarrow$ Action $\rightarrow$ Reward $\rightarrow$ Next State**.
* **Discounting:** We discount future rewards (using a factor $\gamma$) because immediate rewards are more certain than long-term future rewards.
* **The Policy ($\pi$):** The "brain" of the agent. It maps a given **State** to an **Action**. The goal is to find the *optimal policy* $\pi^*$ that maximizes the return.

### Approaches to finding the Optimal Policy:
1.  **Policy-Based Methods:** Training the policy directly (learning *which action* to take).
2.  **Value-Based Methods:** Training a value function (learning *how valuable* a state is) and acting to reach the most valuable states.

* **Deep RL:** Introduces **Deep Neural Networks** to approximate the policy or value functions, allowing agents to solve complex problems with high-dimensional state spaces.

## Install dependencies and create a virtual screen 🔽

The first step is to install the dependencies, we’ll install multiple ones.

- `gymnasium[box2d]`: Contains the LunarLander-v3 environment 🌛
- `stable-baselines3[extra]`: The deep reinforcement learning library.
- `huggingface_sb3`: Additional code for Stable-baselines3 to load and upload models from the Hugging Face 🤗 Hub.

In [2]:
!apt install swig cmake

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 41 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 3s (384 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubunt

In [3]:
!pip install gymnasium[box2d] stable-baselines3 huggingface_sb3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 27.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 20.6 MB/s eta 0:00:00
  Created wheel for box2d-py: filename=box2d_py-2.3.5-cp312-cp312-linux_x86_64.whl size=2381960 sha256=55de2e6ed082f3720766834e9f996318003444e933fac2cd05e8af9f8463bb3b
  Stored in directory: /root/.cache/pip/wheels/2a/e9/60/774da0bcd07f7dc7761a8590fa2d065e4069568e78dcdc3318
Successfully built box2d-py


## 🖥️ System Setup: Virtual Display & Dependencies

Since Google Colab runs on a headless server (without a physical monitor), I need to configure a **virtual display** to render the environment and record the agent's replay videos.

**Steps implemented below:**
1.  **Install Dependencies:** Libraries for rendering (`python-opengl`, `xvfb`) and video processing (`ffmpeg`).
2.  **Runtime Restart:** A necessary step to reload system libraries so the new drivers work correctly.
3.  **Virtual Screen:** Initializing `pyvirtualdisplay` to capture frames.

In [4]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [345 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,227 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe

In [ ]:
import os
# Force a runtime restart to ensure newly installed system libraries (OpenGL) are loaded.
# The notebook will disconnect briefly - this is expected behavior.
os.kill(os.getpid(), 9)

In [1]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 📦 Importing Dependencies

Here I import the necessary libraries to build the RL pipeline:

* **Gymnasium**: To instantiate the `LunarLander-v3` environment.
* **Stable-Baselines3 (SB3)**: To use the PPO algorithm and evaluation metrics.
* **Hugging Face Hub**: To login and push the trained model to the model registry (MLOps).

You can see here all the Deep reinforcement Learning models available here👉 https://huggingface.co/models?pipeline_tag=reinforcement-learning&sort=downloads

In [2]:
import gymnasium

# Hugging Face Hub integration (saving models)
from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

# RL Algorithm and Utilities
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

## 🌍 The Environment: Gymnasium

To simulate the **Lunar Lander** task, I utilize the **Gymnasium** library (formerly OpenAI Gym, now maintained by the Farama Foundation).

It provides a standardized interface for the Reinforcement Learning loop, handling the interaction between the **Agent** and the **Environment**:

1.  **Observation ($S_t$):** The environment gives the agent a state (e.g., lander coordinates, speed).
2.  **Action ($A_t$):** The agent chooses an action (e.g., fire left engine).
3.  **Reward ($R_t$):** The environment returns a numerical reward based on the action's quality.

### The RL Loop Visualization
The process follows this continuous cycle:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/RL_process_game.jpg" alt="The RL process" width="100%">

## 🔄 Implementing the RL Loop with Gymnasium

In this section, I simulate the agent-environment interaction loop using the **Gymnasium API**.

The standard workflow maps theoretical concepts to Python functions as follows:

### 1. The Interaction Cycle

At each timestep:
1.  **Initialize:** Create the environment using `gymnasium.make()`.
2.  **Reset:** Start a new episode with `observation, info = env.reset()`.
3.  **Act & Update:** Call `env.step(action)` to execute an action.

### 2. Understanding `env.step()` Outputs
The `step()` method is the core of the simulation. It returns a tuple of 5 values that I use to guide the training:

* **`observation` (Object):** The new state ($S_{t+1}$). In Lunar Lander, this includes coordinates, velocity, and angle.
* **`reward` (Float):** The immediate feedback ($R_{t+1}$). Positive for landing safely, negative for crashing or using engines.
* **`terminated` (Bool):** `True` if the agent reached a terminal state (Crashed or Landed).
* **`truncated` (Bool):** `True` if the episode ended due to a time limit or out-of-bounds (not natural completion).
* **`info` (Dict):** Auxiliary diagnostic information.

For more explanations check this 👉 https://gymnasium.farama.org/api/env/#gymnasium.Env.step


**Let's look at an example!**

In [3]:
import gymnasium as gym

# First, we create our environment called LunarLander-v3
env = gym.make("LunarLander-v3")

# Then we reset this environment
observation, info = env.reset()

for _ in range(20):
  # Take a random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # Do this action in the environment and get
  # next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # If the game is terminated (in our case we land, crashed) or truncated (timeout)
  if terminated or truncated:
      # Reset the environment
      print("Environment is reset")
      observation, info = env.reset()

env.close()

Action taken: 3
Action taken: 2
Action taken: 1
Action taken: 1
Action taken: 3
Action taken: 1
Action taken: 2
Action taken: 3
Action taken: 0
Action taken: 3
Action taken: 0
Action taken: 0
Action taken: 2
Action taken: 3
Action taken: 1
Action taken: 1
Action taken: 2
Action taken: 2
Action taken: 2
Action taken: 0


/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

## Create the LunarLander environment 🌛 and understand how it works

### [The environment 🎮](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

In this first tutorial, we’re going to train our agent, a [Lunar Lander](https://gymnasium.farama.org/environments/box2d/lunar_lander/), **to land correctly on the moon**. To do that, the agent needs to learn **to adapt its speed and position (horizontal, vertical, and angular) to land correctly.**

---


💡 A good habit when start to use an environment is to check its documentation

👉 https://gymnasium.farama.org/environments/box2d/lunar_lander/

---


Let's see what the Environment looks like:


In [4]:
# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("LunarLander-v3")
env.reset()

print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [ 0.02319365  1.6725909  -8.655326   -7.507785   -1.9229051  -4.471808
  0.2057789   0.22922187]


We see with `Observation Space Shape (8,)` that the observation is a vector of size 8, where each value contains different information about the lander:
- Horizontal pad coordinate (x)
- Vertical pad coordinate (y)
- Horizontal speed (x)
- Vertical speed (y)
- Angle
- Angular speed
- If the left leg contact point has touched the land (boolean)
- If the right leg contact point has touched the land (boolean)


In [5]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 2


The action space (the set of possible actions the agent can take) is discrete with 4 actions available 🎮:

- Action 0: Do nothing,
- Action 1: Fire left orientation engine,
- Action 2: Fire the main engine,
- Action 3: Fire right orientation engine.

Reward function (the function that will give a reward at each timestep) 💰:

After every step a reward is granted. The total reward of an episode is the **sum of the rewards for all the steps within that episode**.

For each step, the reward:

- Is increased/decreased the closer/further the lander is to the landing pad.
-  Is increased/decreased the slower/faster the lander is moving.
- Is decreased the more the lander is tilted (angle not horizontal).
- Is increased by 10 points for each leg that is in contact with the ground.
- Is decreased by 0.03 points each frame a side engine is firing.
- Is decreased by 0.3 points each frame the main engine is firing.

The episode receive an **additional reward of -100 or +100 points for crashing or landing safely respectively.**

An episode is **considered a solution if it scores at least 200 points.**

#### Vectorized Environment

- We create a vectorized environment (a method for stacking multiple independent environments into a single environment) of 16 environments, this way, **we'll have more diverse experiences during the training.**

In [6]:
# Створюємо середовище
env = make_vec_env( "LunarLander-v3" , n_envs= 16 )

## Create the Model 🤖
- We have studied our environment and we understood the problem: **being able to land the Lunar Lander to the Landing Pad correctly by controlling left, right and main orientation engine**. Now let's build the algorithm we're going to use to solve this Problem 🚀.

- To do so, we're going to use our first Deep RL library, [Stable Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/en/master/).

- SB3 is a set of **reliable implementations of reinforcement learning algorithms in PyTorch**.



---

💡 A good habit when using a new library is to dive first on the documentation: https://stable-baselines3.readthedocs.io/en/master/ and then try some tutorials.

----

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/sb3.png" alt="Stable Baselines3">

## 🧠 Agent Training: Proximal Policy Optimization (PPO)

To solve this problem, I utilize the **PPO (Proximal Policy Optimization)** algorithm from the Stable-Baselines3 library. PPO is currently considered a State-of-the-Art (SOTA) method due to its balance between ease of implementation, sample efficiency, and ease of tuning.

### Architecture: Actor-Critic
PPO is a hybrid method that combines two key concepts:
1.  **Policy-Based (The Actor):** Learns a probability distribution over actions (controls *what to do*).
2.  **Value-Based (The Critic):** Learns an action-value function to estimate the expected return (judges *how good* the action was).



### ⚙️ Hyperparameters Configuration
* **Policy Network:** `MlpPolicy` (Multi-Layer Perceptron). I use standard dense neural networks because the input state is a vector of numbers (coordinates, velocity), not an image.
* **Total Timesteps:** `200,000` steps (sufficient for the agent to converge on a winning strategy).

```
from stable_baselines3 import PPO
import gymnasium as gym

# 1. Create the environment
# We use LunarLander-v3 as the standard benchmark
env = gym.make('LunarLander-v3')

# 2. Instantiate the Agent
# - MlpPolicy: We use standard fully connected layers (not CNNs)
# - verbose=1: To see training logs in real-time
model = PPO('MlpPolicy', env, verbose=1)

# 3. Train the Agent
print("🚀 Training started...")
model.learn(total_timesteps=200000)
print("✅ Training completed!")
```

In [8]:
# TODO: Define a PPO MlpPolicy architecture
# We use MultiLayerPerceptron (MLPPolicy) because the input is a vector,
# if we had frames as input we would use CnnPolicy

model = PPO(
    policy='MlpPolicy',      # We use a standard Dense Network (MLP) since input is a vector
    env=env,
    n_steps=1024,            # Number of steps to run for each environment per update
    batch_size=64,           # Minibatch size
    n_epochs=4,              # Number of epoch when optimizing the surrogate loss
    gamma=0.999,             # Discount factor (high value = long-term focus)
    gae_lambda=0.98,         # Factor for trade-off of bias vs variance for GAE
    ent_coef=0.01,           # Entropy coefficient (encourages exploration)
    verbose=1                # Print training logs
)

Using cuda device


## 🏋️ Training the Agent

I proceed to train the PPO agent for **1,000,000 timesteps**. This significant number of steps ensures that the agent has enough interaction with the environment to converge on an optimal policy (consistently landing between flags without crashing).

* **Compute Resource:** Executed on **GPU Runtime** for acceleration (approx. 20 mins).
* **Target Metric:** I am monitoring `ep_rew_mean` (Mean Episode Reward). A solved environment typically requires a score of **200+**.

### 💾 Model Serialization
Once training is complete, I save the model weights locally to `ppo-LunarLander-v3.zip` for later evaluation and deployment to the Hub.

In [9]:
# 1. Train the agent
# Total timesteps = 1,000,000 to ensure full convergence
model.learn(total_timesteps=1000000)

# 2. Save the trained model
# This creates a .zip file containing the policy and value networks
model_name = "ppo-LunarLander-v2"
model.save(model_name)

print(f"✅ Model saved as '{model_name}.zip'")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 91.6     |
|    ep_rew_mean     | -206     |
| time/              |          |
|    fps             | 3642     |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 16384    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 87.1        |
|    ep_rew_mean          | -144        |
| time/                   |             |
|    fps                  | 2439        |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.009342622 |
|    clip_fraction        | 0.0682      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | -0.00126    |
|    learning_rate        | 0.

## 📈 Model Evaluation

Now that the agent is trained, I need to validate its performance quantitatively using standard Stable-Baselines3 tools.

**Evaluation Protocol:**
1.  **Separate Environment:** I create a new instance of `LunarLander-v3`. It is crucial **not** to use the training environment for evaluation to ensure the metrics reflect true generalization (avoiding overfitting).
2.  **Monitoring:** I wrap the evaluation environment in a [Monitor](https://stable-baselines3.readthedocs.io/en/master/common/monitor.html) to capture specific episode statistics like reward and length.
3.  **Metric:** I utilize the [evaluate_policy](https://stable-baselines3.readthedocs.io/en/master/guide/examples.html#basic-usage-training-saving-loading) method to run the agent for **10 test episodes** and calculate the **Mean Reward** and **Standard Deviation**.

*Target Goal: Mean Reward > 200.*

In [10]:
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# 1. Create evaluation env wrapped in Monitor
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))

# 2. Evaluate
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print(f"📊 Evaluation Results:")
print(f"Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")

eval_env.close()

📊 Evaluation Results:
Mean Reward: 253.71 +/- 19.46


## 🔐 Authentication & Setup

Before deploying the model, I authenticate with the **Hugging Face Hub** using a write-access token. This grants the script permission to create a new repository and push the model artifacts.

In [11]:
notebook_login()
!git config --global credential.helper store

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
from huggingface_hub import HfApi

try:
    # Attempt to retrieve user information using the saved token
    user_info = HfApi().whoami()

    print(f"✅ Login successful! Logged in as: {user_info['name']}")
    print(f"🔑 Access type: {user_info['auth']['type']}")

except Exception as e:
    print(f"❌ Login failed. Please check your token. Error: {e}")

✅ Login successful! Logged in as: MykhailoMatsyshyn
🔑 Access type: access_token


## 🚀 Deployment: Publishing the Agent to the Hub

In this final step, I execute a custom deployment script to publish the trained **PPO Agent** to the Hugging Face Hub.

**The script performs the following automated actions:**
1.  **Model Packaging:** Uses `package_to_hub` to save the trained model (`.zip`), network weights, and hyperparameters.
2.  **Metric Evaluation:** Calculates and logs the final performance (Mean Reward) to the Model Card.
3.  **Visual Validation (Custom Fix):** Since automatic video generation can be unstable with newer Gym versions, the script explicitly records a **replay video** of the agent using `RecordVideo` and uploads it manually to the repository.

**Outcome:**
Upon completion, the model will be available at `https://huggingface.co/{repo_id}` with a playable video preview and evaluation metrics.

In [15]:
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from huggingface_sb3 import package_to_hub
from huggingface_hub import HfApi
from gymnasium.wrappers import RecordVideo
import glob
import os

# --- CONFIGURATION ---
repo_id = "MykhailoMatsyshyn/ppo-LunarLander-v3"
env_id = "LunarLander-v3"
model_architecture = "PPO"
commit_message = "Push LunarLander-v3 model with Video Replay 🚀"

# 1. Create the evaluation environment
# We wrap it in Monitor to calculate metrics (Mean Reward)
eval_env = DummyVecEnv([lambda: Monitor(gym.make(env_id, render_mode="rgb_array"))])

# 2. Upload Model and Metrics to Hugging Face
# Note: This step might print an error regarding video generation.
# Ignore it, we will fix the video manually in the next step.
print(f"📦 Starting model upload to {repo_id}...")

package_to_hub(
    model=model,
    model_name="ppo-LunarLander-v3",
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message
)

print("✅ Model and metrics uploaded successfully.")

# 3. Manual Video Recording & Upload (Fix for missing video)
print("🎥 Recording gameplay video manually...")

# Create a separate environment for recording
video_env = gym.make(env_id, render_mode="rgb_array")
# Record one episode to the "manual-replay" folder
video_env = RecordVideo(video_env, video_folder="manual-replay", name_prefix="replay", disable_logger=True)

obs, _ = video_env.reset()
done = False
while not done:
    # Predict the best action (deterministic=True)
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = video_env.step(action)
    done = terminated or truncated
video_env.close()

# 4. Upload the video file
mp4_files = glob.glob('manual-replay/*.mp4')
if mp4_files:
    video_path = mp4_files[0]
    print(f"📤 Uploading video {video_path} to Hugging Face...")

    try:
        api = HfApi()
        api.upload_file(
            path_or_fileobj=video_path,
            path_in_repo="replay.mp4",  # Filename on the Hub
            repo_id=repo_id,
            repo_type="model",
            commit_message="Add gameplay video (manual fix)"
        )
        print("🎉 SUCCESS! Video uploaded. Check your repository.")
        print(f"Link: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"❌ Error uploading video: {e}")
else:
    print("❌ Error: Video file not found locally.")

📦 Starting model upload to MykhailoMatsyshyn/ppo-LunarLander-v3...
ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /tmp/tmpk6gbzfxo/-step-0-to-step-1000.mp4
Moviepy - Building video /tmp/tmpk6gbzfxo/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpk6gbzfxo/-step-0-to-step-1000.mp4



Moviepy - Done !
Moviepy - video ready /tmp/tmpk6gbzfxo/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo MykhailoMatsyshyn/ppo-LunarLander-v3 to the Hugging Face
Hub


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r-v3/policy.optimizer.pth: 100%|##########| 88.7kB / 88.7kB            

  ...LunarLander-v3/policy.pth: 100%|##########| 44.1kB / 44.1kB            

  ...-v3/pytorch_variables.pth: 100%|##########| 1.26kB / 1.26kB            

  ...0f/ppo-LunarLander-v3.zip:  14%|#4        | 21.4kB /  150kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/MykhailoMatsyshyn/ppo-LunarLander-v3/tree/main/
✅ Model and metrics uploaded successfully.
🎥 Recording gameplay video manually...
📤 Uploading video manual-replay/replay-episode-0.mp4 to Hugging Face...
🎉 SUCCESS! Video uploaded. Check your repository.
Link: https://huggingface.co/MykhailoMatsyshyn/ppo-LunarLander-v3


## 🎥 Visual Validation: Multi-Episode Replay

To demonstrate the agent's ability to generalize to different terrain configurations, I record **5 distinct episodes**.

**Process:**
1.  **Recording:** The agent plays 10 full games on randomly generated maps.
2.  **Merging:** Using `FFmpeg`, I concatenate these episodes into a single continuous video file (`merged_replay.mp4`).
3.  **Uploading:** The final composite video is uploaded to the Hugging Face Hub, replacing the default replay.

In [22]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import glob
import os
import shutil
from huggingface_hub import HfApi

# --- CONFIGURATION ---
N_EPISODES = 10  # Number of episodes to combine (shows agent generalization)
REPO_ID = "MykhailoMatsyshyn/ppo-LunarLander-v3"
TEMP_FOLDER = "temp_episodes"
FINAL_VIDEO_NAME = "merged_replay.mp4"
ENV_ID = "LunarLander-v3"

print(f"🎬 Starting production of a {N_EPISODES}-episode series...")

# --- STEP 1: Record Individual Episodes ---
# Clean up temporary folder if it exists
if os.path.exists(TEMP_FOLDER):
    shutil.rmtree(TEMP_FOLDER)
os.makedirs(TEMP_FOLDER, exist_ok=True)

# Create environment
env = gym.make(ENV_ID, render_mode="rgb_array")
# Configure recording: record EVERY episode
env = RecordVideo(env, video_folder=TEMP_FOLDER, name_prefix="ep", episode_trigger=lambda x: True, disable_logger=True)

for i in range(N_EPISODES):
    obs, _ = env.reset()
    done = False
    print(f"🎥 Recording episode {i+1}/{N_EPISODES} (generating new terrain)...")

    while not done:
        # Predict best action (deterministic=True)
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
env.close()

# --- STEP 2: Merge Videos using FFmpeg ---
print("\n🎞️ Merging episodes into one final video...")

# Find all recorded mp4 files and sort them
mp4_files = sorted(glob.glob(f'{TEMP_FOLDER}/*.mp4'))

if not mp4_files:
    print("❌ Error: No videos recorded.")
else:
    # Create a list file for FFmpeg
    list_file = "video_list.txt"
    with open(list_file, 'w') as f:
        for mp4 in mp4_files:
            # FFmpeg requires absolute paths
            abs_path = os.path.abspath(mp4)
            f.write(f"file '{abs_path}'\n")

    # Run FFmpeg command to concatenate videos without re-encoding (very fast)
    # -f concat: concatenation mode
    # -safe 0: allow unsafe file paths (required for Colab)
    # -c copy: copy streams directly
    concat_command = f"ffmpeg -y -f concat -safe 0 -i {list_file} -c copy {FINAL_VIDEO_NAME} -loglevel error"
    os.system(concat_command)

    if os.path.exists(FINAL_VIDEO_NAME):
        print(f"✅ Video merged successfully: {FINAL_VIDEO_NAME}")

        # --- STEP 3: Upload to Hugging Face Hub ---
        print(f"\n📤 Uploading final video to {REPO_ID}...")
        try:
            api = HfApi()
            api.upload_file(
                path_or_fileobj=FINAL_VIDEO_NAME,
                path_in_repo="replay.mp4", # Filename on the Hub
                repo_id=REPO_ID,
                repo_type="model",
                commit_message=f"Add composite video of {N_EPISODES} different landings 🎬"
            )
            print("🎉 SUCCESS! Video uploaded.")
            print(f"Check your model page: https://huggingface.co/{REPO_ID}")
            print("(Note: It may take a minute for the video to appear on the Hub)")
        except Exception as e:
            print(f"❌ Error uploading video: {e}")
    else:
        print("❌ Error: FFmpeg failed to merge videos.")

# Cleanup temporary files
if os.path.exists(TEMP_FOLDER): shutil.rmtree(TEMP_FOLDER)
if os.path.exists(list_file): os.remove(list_file)
# if os.path.exists(FINAL_VIDEO_NAME): os.remove(FINAL_VIDEO_NAME) # Optional: keep the file to download

🎬 Starting production of a 10-episode series...
🎥 Recording episode 1/10 (generating new terrain)...
🎥 Recording episode 2/10 (generating new terrain)...
🎥 Recording episode 3/10 (generating new terrain)...
🎥 Recording episode 4/10 (generating new terrain)...
🎥 Recording episode 5/10 (generating new terrain)...
🎥 Recording episode 6/10 (generating new terrain)...
🎥 Recording episode 7/10 (generating new terrain)...
🎥 Recording episode 8/10 (generating new terrain)...
🎥 Recording episode 9/10 (generating new terrain)...
🎥 Recording episode 10/10 (generating new terrain)...

🎞️ Merging episodes into one final video...
✅ Video merged successfully: merged_replay.mp4

📤 Uploading final video to MykhailoMatsyshyn/ppo-LunarLander-v3...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  merged_replay.mp4           : 100%|##########|  256kB /  256kB            

🎉 SUCCESS! Video uploaded.
Check your model page: https://huggingface.co/MykhailoMatsyshyn/ppo-LunarLander-v3
(Note: It may take a minute for the video to appear on the Hub)


## 📥 Loading and Evaluating a Pre-trained Model

To benchmark my results or use transfer learning, I can download existing models from the Hugging Face Hub.

**Addressing Compatibility Issues:**
1.  **Shimmy:** Since many models on the Hub were trained with the legacy `Gym` library, I install the [Shimmy](https://github.com/Farama-Foundation/Shimmy) API conversion tool to make them compatible with my current `Gymnasium` setup.
2.  **Serialization Handling:** To avoid version mismatches (e.g., Python 3.7 vs 3.10 pickle protocols), I pass `custom_objects` during loading. This ensures that serialized functions like the learning rate schedule do not crash the loading process.

**Execution:**
Below, I load a community-trained agent (`Classroom-workshop`) and evaluate its performance specifically in the updated **`LunarLander-v3`** environment.

In [16]:
!pip install shimmy

In [17]:
from huggingface_sb3 import load_from_hub
repo_id = "Classroom-workshop/assignment2-omar" # The repo_id
filename = "ppo-LunarLander-v2.zip" # The model filename.zip

# When the model was trained on Python 3.8 the pickle protocol is 5
# But Python 3.6, 3.7 use protocol 4
# In order to get compatibility we need to:
# 1. Install pickle5 (we done it at the beginning of the colab)
# 2. Create a custom empty object we pass as parameter to PPO.load()
custom_objects = {
            "learning_rate": 0.0,
            "lr_schedule": lambda _: 0.0,
            "clip_range": lambda _: 0.0,
}

checkpoint = load_from_hub(repo_id, filename)
model = PPO.load(checkpoint, custom_objects=custom_objects, print_system_info=True)

ppo-LunarLander-v2.zip:   0%|          | 0.00/146k [00:00<?, ?B/s]

== CURRENT SYSTEM INFO ==
- OS: Linux-6.6.105+-x86_64-with-glibc2.35 # 1 SMP Thu Oct  2 10:42:05 UTC 2025
- Python: 3.12.12
- Stable-Baselines3: 2.7.1
- PyTorch: 2.9.0+cu126
- GPU Enabled: True
- Numpy: 2.0.2
- Cloudpickle: 3.1.2
- Gymnasium: 1.2.2
- OpenAI Gym: 0.25.2

== SAVED MODEL SYSTEM INFO ==
OS: Linux-5.4.188+-x86_64-with-Ubuntu-18.04-bionic #1 SMP Sun Apr 24 10:03:06 PDT 2022
Python: 3.7.13
Stable-Baselines3: 1.5.0
PyTorch: 1.11.0+cu113
GPU Enabled: True
Numpy: 1.21.6
Gym: 0.21.0



/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/save_util.py:165: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  deserialized_object = cloudpickle.loads(base64_object)
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:95: UserWarning: You loaded a model that was trained using OpenAI Gym. We strongly recommend transitioning to Gymnasium by saving that model again.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algori

In [19]:
#@title
eval_env = Monitor(gym.make("LunarLander-v3"))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=290.86 +/- 19.83776549323971


# 🏁 Conclusion & Key Takeaways

In this project, I successfully implemented a complete **Deep Reinforcement Learning pipeline** to solve the Lunar Lander environment.

### 🎯 Key Achievements:
1.  **Environment Solved:** Trained a PPO agent that consistently achieves a Mean Reward of **>200**, demonstrating stable and precise landing capabilities.
2.  **Algorithm Mastery:** configured the **Proximal Policy Optimization (PPO)** algorithm, understanding the balance between exploration and exploitation.
3.  **MLOps & Deployment:** Successfully deployed the trained model to the **Hugging Face Hub**.
4.  **Technical Problem Solving:** Overcame library compatibility issues (Gymnasium vs. Legacy Gym) by implementing custom scripts for **video recording** and using **Shimmy** for loading legacy models.

### 🔗 Resources:
* **My Trained Model:** [MykhailoMatsyshyn/ppo-LunarLander-v3](https://huggingface.co/MykhailoMatsyshyn/ppo-LunarLander-v3)
